# Module 04 · Unit 01
# Graph Definitions

| | |
|---|---|
| **Estimated time** | 55–65 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Attack Graphs & Reachability \[AG\] · Neural Network Architecture \[NN\] |
| **Refresher** | Module 03 · Unit 02 — Relations and Functions |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Define a graph formally as a pair $(V, E)$ and recognise graphs in real systems
2. Compute **degree**, **adjacency**, and **neighbourhood** for any vertex
3. Distinguish **walks**, **paths**, **trails**, and **cycles**
4. Determine whether a graph is **connected** and identify its **connected components**
5. Recognise **bipartite** and **weighted** graphs and explain their security uses
6. Use NetworkX to build, query, and visualise graphs from real system data


## 🔣 Symbol Primer

Graph theory has its own compact notation. The most important thing to notice:
a graph is built directly on the set and relation machinery from Module 03.

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $G = (V, E)$ | Graph | "graph $G$ with vertices $V$ and edges $E$" | A set of nodes and a set of connections |
| $V$ | Vertex set | "vertices" or "nodes" | The objects in the graph |
| $E$ | Edge set | "edges" | The connections — pairs of vertices |
| $\{u, v\}$ | Undirected edge | "edge between $u$ and $v$" | A symmetric connection: $\{u,v\} = \{v,u\}$ |
| $n = \|V\|$ | Order | "order of $G$" | Number of vertices |
| $m = \|E\|$ | Size | "size of $G$" | Number of edges |
| $\deg(v)$ | Degree | "degree of $v$" | Number of edges incident to $v$ |
| $N(v)$ | Neighbourhood | "neighbours of $v$" | $\{u \in V \mid \{u,v\} \in E\}$ |
| $G[S]$ | Induced subgraph | "$G$ induced by $S$" | The subgraph on vertex subset $S \subseteq V$ |
| $w(e)$ | Edge weight | "weight of edge $e$" | A numerical value assigned to an edge |

> **The connection to Module 03.** A graph is a set $V$ with a binary relation
> $E \subseteq V \times V$ (symmetric, irreflexive for simple undirected graphs).
> Everything you know about relations applies directly. The **adjacency matrix**
> of a graph is exactly the Boolean matrix representation of this relation.

---


## 1 · What Is a Graph? You Already Know One

Before the formal definition, consider what you already work with every day:

- A **computer network** — servers are vertices, cables are edges
- A **social network** — people are vertices, friendships are edges
- A **neural network** — neurons are vertices, weighted connections are edges
- A **road map** — cities are vertices, roads are edges
- A **dependency tree** — packages are vertices, "requires" relationships are edges
- An **attack path** — systems are vertices, exploitable connections are edges

All of these are graphs. The formal definition simply names the pieces.

### Formal Definition

A **simple undirected graph** $G = (V, E)$ consists of:

- A non-empty **vertex set** $V$ — a finite set of objects (also called *nodes*)
- An **edge set** $E$ — a set of 2-element subsets of $V$

$$E \subseteq \binom{V}{2} = \{\{u, v\} \mid u, v \in V,\; u \neq v\}$$

*Simple* means no self-loops ($\{v,v\}$) and no multiple edges between the same pair.
An **undirected** edge $\{u, v\}$ has no preferred direction — it is symmetric.

### Example

$$V = \{A, B, C, D\} \qquad E = \{\{A,B\},\, \{A,C\},\, \{B,C\},\, \{C,D\}\}$$

This graph has **4 vertices** (order $n = 4$) and **4 edges** (size $m = 4$).
It represents a small network where $A$, $B$, and $C$ are all interconnected
and $D$ is connected only to $C$ — a classic "peripheral node" pattern.

### The Adjacency Matrix

The **adjacency matrix** $\mathbf{A}$ of graph $G$ is the $n \times n$ Boolean matrix
where $\mathbf{A}_{ij} = 1$ if $\{i, j\} \in E$ and $0$ otherwise.

For an undirected graph, $\mathbf{A}$ is **symmetric**: $\mathbf{A}_{ij} = \mathbf{A}_{ji}$.
This is the binary relation matrix from Module 03, restricted to symmetric relations.


## 2 · Degree and Neighbourhood

The **degree** of a vertex $v$ is the number of edges incident to it — the number
of direct connections it has:

$$\deg(v) = |\{e \in E \mid v \in e\}| = |N(v)|$$

where $N(v) = \{u \in V \mid \{u,v\} \in E\}$ is the **neighbourhood** of $v$.

### The Handshaking Lemma

$$\sum_{v \in V} \deg(v) = 2|E|$$

Every edge contributes exactly 2 to the total degree count — once for each endpoint.
This means the sum of all degrees is always even, and the number of odd-degree
vertices is always even.

### Security Interpretation of Degree

| Context | High degree means... | Security implication |
|---|---|---|
| Network graph | Many direct connections | High-value target; wide blast radius if compromised |
| Dependency graph | Many dependents | Single point of failure — log4j pattern |
| Attack graph | Many reachable systems | Pivot point for lateral movement |
| Social graph | Many connections | High-influence account — impersonation target |

**The log4j pattern.** When log4j was found to have a critical RCE vulnerability
in late 2021, its high degree in the dependency graph — thousands of projects
directly or transitively depending on it — was precisely what made the blast radius
so large. Degree in the dependency graph is a direct measure of blast radius.


## 3 · Walks, Paths, Trails, and Cycles

Moving through a graph follows precise rules. The terminology matters because
different security analyses require different notions of "movement."

### Definitions

A **walk** of length $k$ from $v_0$ to $v_k$ is a sequence of vertices:

$$v_0,\, v_1,\, v_2,\, \ldots,\, v_k$$

where $\{v_i, v_{i+1}\} \in E$ for all $i$. Vertices and edges may repeat.

A **trail** is a walk in which no **edge** is repeated (but vertices may repeat).

A **path** is a walk in which no **vertex** is repeated — the cleanest notion of
"getting from $u$ to $v$ without backtracking."

A **cycle** is a closed path: $v_0 = v_k$ and all intermediate vertices are distinct.

| Term | Vertices repeat? | Edges repeat? |
|---|:---:|:---:|
| Walk | ✓ | ✓ |
| Trail | ✓ | ✗ |
| Path | ✗ | ✗ |
| Cycle | only start=end | ✗ |

### Security Relevance

**Attack paths.** An attacker traversing a network follows a path — they move
through distinct systems without revisiting (in the simplest model). The minimum
number of hops between an entry point and a target is the length of the **shortest
path** — a key metric we will compute in Module 05.

**Cycles and infinite loops.** A cycle in a dependency graph means circular
dependency — package A requires B which requires A. A cycle in a protocol state
machine can mean an infinite loop or a replay attack. Detecting cycles is one of
the primary reasons to analyse graph structure formally.


## 4 · Connectivity and Connected Components

A graph $G$ is **connected** if there is a path between every pair of vertices:

$$\forall u, v \in V,\; \exists \text{ a path from } u \text{ to } v$$

This is a $\forall\exists$ statement — exactly the predicate logic from Module 02,
now applied to graph structure.

A **connected component** of $G$ is a maximal connected subgraph — a subset of
vertices where all are reachable from each other, and no more vertices can be added
while preserving that property.

### Formal Definition of a Component

A connected component is an equivalence class of the relation $\sim$ defined by:
$$u \sim v \;\Longleftrightarrow\; \exists \text{ a path from } u \text{ to } v$$

This is the equivalence relation from Module 03 — reflexive (path of length 0),
symmetric (undirected edges), and transitive (concatenate paths). The components
are its equivalence classes. Graph connectivity is set-relation theory applied
to paths.

### Security Interpretation

| Scenario | Connected | Disconnected |
|---|---|---|
| Enterprise network | All hosts reachable — easy lateral movement | Segments isolated — attacker confined |
| Dependency graph | All packages in one component — transitive blast radius | Isolated components — contained failure |
| Social graph | Fully reachable — information spreads everywhere | Isolated clusters — contained compromise |

**Network segmentation** is the deliberate act of making a network graph
*disconnected* — breaking it into components that cannot reach each other.
The security goal is to ensure that no attacker-reachable component contains
high-value targets.


## 5 · Special Graph Types

### Bipartite Graphs

A graph $G = (V, E)$ is **bipartite** if $V$ can be partitioned into two disjoint
sets $V = A \cup B$ (with $A \cap B = \emptyset$) such that every edge connects
a vertex in $A$ to a vertex in $B$ — no edges within $A$ or within $B$.

$$E \subseteq \{\{a, b\} \mid a \in A,\; b \in B\}$$

**Security examples:**
- **Access control:** $A$ = Users, $B$ = Resources. Every edge is a permission.
  The access control relation *is* a bipartite graph — the same structure we
  studied as a binary relation in Module 03.
- **Transformer attention:** $A$ = Query vectors, $B$ = Key vectors. Each edge
  is an attention weight — how much each query "attends to" each key.
- **Credential assignment:** $A$ = Users, $B$ = API keys. Each edge is an assignment.

### Weighted Graphs

A **weighted graph** assigns a numerical value $w: E \to \mathbb{R}$ to each edge.

**Security examples:**
- **Attack graphs:** edge weight = difficulty of exploiting a vulnerability
  (CVSS score, time, skill level required)
- **Network graphs:** edge weight = latency, bandwidth, or trust score
- **Attention graphs:** edge weight = softmax attention probability

Weighted graphs are the natural representation of *quantified risk* — we will
use them heavily in Module 05 when computing minimum-cost attack paths.

### Complete Graphs

$K_n$ is the **complete graph** on $n$ vertices — every possible edge exists.
$|E| = \binom{n}{2} = \frac{n(n-1)}{2}$.

A fully connected neural network layer, where every neuron connects to every
neuron in the next layer, is a complete bipartite graph $K_{n,m}$.


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AG\] \[NN\]

| Graph concept | \[AG\] Attack Graphs | \[NN\] Neural Architecture |
|---|---|---|
| **Vertex** | Host, service, or vulnerability | Neuron, layer, or operation |
| **Edge** | Exploitable connection | Weighted synaptic connection |
| **Degree** | Blast radius indicator | Fan-in / fan-out of a layer |
| **Path** | Attack route from entry to target | Forward pass computation chain |
| **Cycle** | Lateral movement loop | Recurrent connection (RNN) |
| **Connected component** | Isolated network segment | Independent sub-network |
| **Bipartite graph** | User ↔ Resource access matrix | Query ↔ Key attention map |
| **Weighted graph** | Exploit difficulty scores | Attention weights, synaptic strengths |
| **Connectivity** | Can attacker reach target? | Can gradient flow from loss to weights? |

**The core insight of Module 04.** A graph is the formal structure that
captures *reachability* — whether you can get from one place to another.
In security, reachability is the fundamental question: can an attacker reach
the target from their entry point? In neural networks, the same question is:
can a gradient signal reach a weight from the loss function?

The mathematics is identical. The stakes are different.

---


## Python: Visualization & Verification

**1 · Building and Querying Graphs** — construct a network graph from real-looking
data, compute degree, adjacency, and neighbourhood, and identify high-degree
vertices as security risk indicators.

**2 · Paths, Cycles, and Connectivity** — find paths between vertices, detect
cycles, identify connected components, and visualise network segmentation
as deliberate disconnection.

**3 · Bipartite Access Control Graph** — represent the access control relation
from Module 03 as a bipartite graph; visualise it; compute the degree sequence
on both sides to identify over-provisioned users and single-point-of-access resources.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · Building and Querying a Network Graph

We model a small enterprise network as a graph. Each node is a host or service;
each edge is a direct network connection. We compute degree, adjacency, and
neighbourhood for every vertex — then flag high-degree nodes as priority
hardening targets.


In [ ]:
# ── 1 · Network Graph — Build, Query, Visualise ───────────────────────────────

G = nx.Graph()

# Vertices: hosts and services in a small enterprise network
hosts = [
    'internet',        # external entry point
    'firewall',        # perimeter
    'web_server',      # DMZ
    'api_server',      # DMZ
    'load_balancer',   # DMZ
    'db_primary',      # internal
    'db_replica',      # internal
    'admin_workstation', # internal
    'dev_laptop',      # internal
    'log_server',      # internal
    'backup_server',   # internal
]
G.add_nodes_from(hosts)

# Edges: direct network connections
edges = [
    ('internet',        'firewall'),
    ('firewall',        'web_server'),
    ('firewall',        'api_server'),
    ('firewall',        'load_balancer'),
    ('web_server',      'load_balancer'),
    ('api_server',      'load_balancer'),
    ('load_balancer',   'db_primary'),
    ('api_server',      'db_primary'),
    ('db_primary',      'db_replica'),
    ('db_primary',      'backup_server'),
    ('admin_workstation','db_primary'),
    ('admin_workstation','api_server'),
    ('admin_workstation','log_server'),
    ('dev_laptop',      'api_server'),
    ('dev_laptop',      'log_server'),
    ('log_server',      'backup_server'),
]
G.add_edges_from(edges)

# ── Graph properties ───────────────────────────────────────────────────────────
print(f"Graph G = (V, E)")
print(f"  Order  |V| = {G.number_of_nodes()} vertices")
print(f"  Size   |E| = {G.number_of_edges()} edges")
print(f"  Connected:  {nx.is_connected(G)}")
print(f"  Handshaking lemma: Σ deg(v) = {sum(dict(G.degree()).values())} = 2 × {G.number_of_edges()}")
print()

# Degree table
degrees = sorted(G.degree(), key=lambda x: x[1], reverse=True)
print(f"{'Vertex':<22} {'deg(v)':>6}  {'N(v)'}")
print("─" * 65)
for v, d in degrees:
    print(f"  {v:<20} {d:>6}  {sorted(G.neighbors(v))}")

# ── Visualise ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))
pos = nx.spring_layout(G, seed=42, k=2.2)

# Colour by degree — higher degree = warmer colour
deg_vals = dict(G.degree())
max_deg  = max(deg_vals.values())
node_colors = [
    TS_RED   if deg_vals[v] >= 4 else
    TS_AMBER if deg_vals[v] >= 3 else
    TS_BLUE  if deg_vals[v] >= 2 else
    TS_GRAY
    for v in G.nodes()
]
node_sizes = [300 + 200 * deg_vals[v] for v in G.nodes()]

nx.draw_networkx_edges(G, pos, ax=ax, edge_color=TS_GRAY,
                       alpha=0.6, width=1.8)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors,
                       node_size=node_sizes, alpha=0.92)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8,
                        font_color='white', font_weight='bold')

# Legend
legend_elements = [
    mpatches.Patch(color=TS_RED,   label='deg ≥ 4  (critical — harden first)'),
    mpatches.Patch(color=TS_AMBER, label='deg = 3  (high priority)'),
    mpatches.Patch(color=TS_BLUE,  label='deg = 2  (moderate)'),
    mpatches.Patch(color=TS_GRAY,  label='deg = 1  (peripheral)'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)
ax.set_title('Enterprise Network Graph — Node Colour by Degree
'
             'Larger / warmer nodes have more connections — higher blast radius',
             pad=12, fontsize=11)
ax.axis('off')
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The degree table immediately surfaces the highest-risk nodes:

- `db_primary` has the highest degree — it is directly connected to the API server,
  load balancer, admin workstation, replica, and backup. Compromising it exposes
  data, disrupts replication, and potentially reaches the backup. Maximum blast radius.
- `api_server` and `load_balancer` are also high-degree — they sit at the intersection
  of the external-facing and internal layers.
- `internet` has degree 1 — it connects only to the firewall. Good containment.

The Handshaking Lemma check confirms: sum of all degrees = 2 × number of edges.
This is not interesting as a fact — it is a quick sanity check that you have
built the graph correctly. If the sum were odd, you made an error.

**The security principle:** degree is not the only risk factor, but it is the
first one to check. A vulnerability in a degree-1 peripheral node has a bounded
blast radius. A vulnerability in a high-degree internal node can cascade everywhere.


### 2 · Paths, Cycles, and Network Segmentation

We find all paths between the internet entry point and the database, identify
any cycles in the network, and then demonstrate network segmentation by removing
the firewall and observing how the graph splits into components.


In [ ]:
# ── 2 · Paths, Cycles, and Connectivity ──────────────────────────────────────

# All simple paths from internet to db_primary
src, tgt = 'internet', 'db_primary'
all_paths = list(nx.all_simple_paths(G, src, tgt))
all_paths_sorted = sorted(all_paths, key=len)

print(f"Simple paths from '{src}' to '{tgt}':")
for i, p in enumerate(all_paths_sorted):
    print(f"  {i+1}. Length {len(p)-1}: {' → '.join(p)}")

print(f"\nShortest path length: {len(all_paths_sorted[0])-1} hops")
print(f"Total distinct paths: {len(all_paths)}")

# Cycle detection
print(f"\nCycles in network graph:")
try:
    cycle = nx.find_cycle(G)
    print(f"  Cycle found: {cycle}")
except nx.NetworkXNoCycle:
    print(f"  No cycles detected — this is a tree-like (acyclic) topology.")

# ── Network segmentation: remove firewall, observe components ─────────────────
G_segmented = G.copy()
G_segmented.remove_node('firewall')

components = list(nx.connected_components(G_segmented))
print(f"\nAfter removing 'firewall' (simulating firewall failure/segmentation):")
print(f"  Connected components: {len(components)}")
for i, comp in enumerate(sorted(components, key=len, reverse=True)):
    print(f"  Component {i+1} ({len(comp)} nodes): {sorted(comp)}")

# ── Visualise before/after segmentation ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, graph, title in [
    (axes[0], G,          'Original Network — fully connected'),
    (axes[1], G_segmented,'After removing firewall — segmented'),
]:
    comps = list(nx.connected_components(graph))
    comp_color_map = {}
    palette = [TS_BLUE, TS_AMBER, TS_GREEN, TS_RED, TS_GRAY]
    for ci, comp in enumerate(comps):
        for node in comp:
            comp_color_map[node] = palette[ci % len(palette)]

    pos_g = nx.spring_layout(graph, seed=42, k=2.5)
    colors = [comp_color_map.get(v, TS_GRAY) for v in graph.nodes()]

    nx.draw_networkx_edges(graph, pos_g, ax=ax,
                           edge_color=TS_GRAY, alpha=0.5, width=1.8)
    nx.draw_networkx_nodes(graph, pos_g, ax=ax,
                           node_color=colors, node_size=600, alpha=0.9)
    nx.draw_networkx_labels(graph, pos_g, ax=ax,
                            font_size=7, font_color='white', font_weight='bold')
    ax.set_title(f'{title}\n({len(comps)} component{"s" if len(comps)>1 else ""})',
                 fontsize=10, fontweight='bold', pad=8)
    ax.axis('off')
    ax.set_facecolor('white')

plt.suptitle('Network Segmentation as Graph Disconnection',
             fontsize=12, fontweight='bold', y=1.02)
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()

# Highlight the shortest attack path
print(f"\nShortest attack path (internet → db_primary):")
shortest = all_paths_sorted[0]
print(f"  {' → '.join(shortest)}  ({len(shortest)-1} hops)")
print(f"  Every hop is an opportunity for detection and blocking.")


**What do you see?**

The path analysis shows every distinct route an attacker could take from the
internet entry point to the database — and there are multiple. The shortest path
is the attacker's preferred route; longer paths are fallbacks if the short one
is blocked.

The network has no cycles — it is a tree-like topology (common in well-designed
networks). Cycles would mean multiple routes between internal nodes — sometimes
desired for redundancy, but also meaning more attack paths.

After removing the firewall, the graph splits into two components (colour-coded):
the internet node is isolated, and the internal network becomes a separate component.
This is what effective perimeter security means in graph terms: the firewall
is the **cut vertex** that, when removed, disconnects the external from the internal.
Module 05 will formalise cut vertices and use Dijkstra's algorithm to compute
the minimum-cost attack path with explicit edge weights.


### 3 · Bipartite Access Control Graph

We represent the Module 03 access control relation as a bipartite graph with
$A$ = Users and $B$ = Resources. Each edge is a permission grant. The bipartite
structure makes over-provisioning and single-points-of-access immediately visible
from the degree sequence.


In [ ]:
# ── 3 · Bipartite Access Control Graph ───────────────────────────────────────

users     = ['Alice', 'Bob', 'Carol', 'Dave', 'Eve']
resources = ['payroll', 'source_code', 'logs', 'config', 'public_docs']

# Access relation (same as Module 03 Unit 02)
access_edges = [
    ('Alice','payroll'),('Alice','source_code'),('Alice','logs'),
    ('Alice','config'), ('Alice','public_docs'),
    ('Bob',  'source_code'), ('Bob','logs'), ('Bob','public_docs'),
    ('Carol','payroll'), ('Carol','logs'), ('Carol','public_docs'),
    ('Dave', 'public_docs'),
    ('Eve',  'logs'), ('Eve','source_code'), ('Eve','public_docs'),
]

B = nx.Graph()
B.add_nodes_from(users,     bipartite=0)
B.add_nodes_from(resources, bipartite=1)
B.add_edges_from(access_edges)

# Verify bipartiteness
print(f"Is bipartite: {nx.is_bipartite(B)}")
print(f"  |Users|     = {len(users)}")
print(f"  |Resources| = {len(resources)}")
print(f"  |Edges|     = {B.number_of_edges()} (access grants)")

print(f"\nUser degrees (number of resources accessible):")
for u in users:
    neighbours = sorted(B.neighbors(u))
    flag = '  ⚠ HIGH' if B.degree(u) >= 4 else ''
    print(f"  {u:<8}  deg={B.degree(u)}  {neighbours}{flag}")

print(f"\nResource degrees (number of users with access):")
for r in resources:
    users_with_access = sorted(B.neighbors(r))
    flag = '  ⚠ SINGLE POINT' if B.degree(r) == 1 else ''
    print(f"  {r:<15}  deg={B.degree(r)}  {users_with_access}{flag}")

# ── Bipartite visualisation ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
pos = nx.bipartite_layout(B, users, align='vertical', scale=2)

# Colour nodes
user_colors     = [TS_RED   if B.degree(u) >= 4 else TS_BLUE  for u in users]
resource_colors = [TS_AMBER if B.degree(r) == 1 else TS_GREEN for r in resources]

nx.draw_networkx_edges(B, pos, ax=ax, edge_color=TS_GRAY, alpha=0.5, width=1.5)
nx.draw_networkx_nodes(B, pos, ax=ax,
                       nodelist=users,
                       node_color=user_colors, node_size=900, alpha=0.9)
nx.draw_networkx_nodes(B, pos, ax=ax,
                       nodelist=resources,
                       node_color=resource_colors, node_size=900,
                       node_shape='s', alpha=0.9)
nx.draw_networkx_labels(B, pos, ax=ax, font_size=8,
                        font_color='white', font_weight='bold')

legend_elements = [
    mpatches.Patch(color=TS_RED,   label='User — over-provisioned (deg ≥ 4)'),
    mpatches.Patch(color=TS_BLUE,  label='User — normal access'),
    mpatches.Patch(color=TS_AMBER, label='Resource — single-point-of-access ⚠'),
    mpatches.Patch(color=TS_GREEN, label='Resource — multiple authorised users'),
]
ax.legend(handles=legend_elements, loc='lower center',
          bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=9)
ax.set_title('Bipartite Access Control Graph
'
             'Circles = Users (left)   Squares = Resources (right)   Edges = permissions',
             pad=12, fontsize=11)
ax.axis('off')
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
plt.tight_layout()
plt.show()


**What do you see?**

The bipartite layout separates users (left, circles) from resources (right, squares)
with edges representing granted permissions.

- **Alice** (red — over-provisioned) has degree 5: she can access every resource.
  This is the classic admin over-provisioning pattern — if Alice's account is
  compromised, every resource is exposed.

- **`config`** (amber — single-point-of-access) has degree 1: only Alice can access
  the configuration. This is an availability risk — disable Alice's account and
  no one can change system configuration.

The bipartite degree sequence is a **security audit in one line of code**:
users with high degree are over-provisioned; resources with low degree are availability
risks. Both are immediate action items.

**The connection to Module 03.** This is the exact same access control relation
we stored as a binary relation matrix in Module 03 Unit 02. Representing it as a
bipartite graph adds no new information — but the visual structure surfaces the
security findings immediately, in a way the matrix did not.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. HANDSHAKING LEMMA PROOF CHECK
#    Generate 10 random graphs using nx.erdos_renyi_graph(n=15, p=0.3, seed=i).
#    For each, verify: sum of degrees == 2 * number of edges.
#    Print a confirmation for each. Does it ever fail?
#
# 2. CUT VERTEX ANALYSIS
#    NetworkX has nx.articulation_points(G) which finds all cut vertices —
#    vertices whose removal disconnects the graph.
#    Apply it to the enterprise network G.
#    Which nodes are cut vertices? What does this mean for network resilience?
#    How does this relate to the firewall removal we demonstrated?
#
# 3. WEIGHTED BIPARTITE GRAPH
#    Extend the access control graph with edge weights representing
#    "permission sensitivity" (1=low, 3=high):
#      payroll=3, source_code=3, config=3, logs=2, public_docs=1
#    Compute the "total risk exposure" of each user as the sum of edge weights.
#    Who has the highest risk exposure? Plot the weighted bipartite graph
#    with edge thickness proportional to weight.

# Your work here:


---

## Summary

| Concept | Definition | Security meaning |
|---|---|---|
| **Graph** $G=(V,E)$ | Vertices + edges | Any system with objects and connections |
| **Degree** $\deg(v)$ | Number of edges at $v$ | Blast radius indicator; hardening priority |
| **Neighbourhood** $N(v)$ | Set of adjacent vertices | Direct dependencies; direct exposure |
| **Walk / Path / Cycle** | Sequences of connected vertices | Attack routes; dependency chains; loops |
| **Connected** | $\forall u,v,\; \exists$ path $u \to v$ | Every node reachable — no segmentation |
| **Component** | Maximal connected subgraph | Isolated network segment |
| **Bipartite** | Two-part vertex set, edges only across | Access control: users ↔ resources |
| **Weighted** | Numerical values on edges | Exploit difficulty; attention weights; trust |

---

## Up Next

**Module 04 · Unit 02 — Directed Graphs and DAGs**

We add direction to edges. In an undirected graph, a connection between $u$ and $v$
is symmetric. In a directed graph, an edge $(u, v)$ means $u$ *points to* $v$ — not
the same as $v$ pointing to $u$. Direction is what makes computation graphs,
dependency graphs, and attack graphs precise — and what makes the concept of a
**DAG** (Directed Acyclic Graph) the native structure of neural network forward passes.

$\rightarrow$ `modules/module-04/unit-02-directed-graphs-dags.ipynb`
